# Just an example.You can alter sample code anywhere.

## Mount your google drive

In [1]:
#from google.colab import drive
#drive.mount('/content/drive')

In [2]:
# You need to modify this part to the directory where your code is located

#%cd "/content/drive/MyDrive/Lab01/"
%cd "D:\USER\Desktop\DL\2025_DL_Lab01\Lab01"

D:\USER\Desktop\DL\2025_DL_Lab01\Lab01


## Import packages

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import pandas as pd

In [4]:
#Fix the random seed
np.random.seed(3645)

## Load the data and label

In [5]:
train_load = np.loadtxt('./data/fmnist-train.csv',delimiter=',',dtype="int")
train_data=train_load[:,1:]
train_label=train_load[:,0]
print("shape of train_data: {}".format(train_data.shape))
print("shape of train_label: {}".format(train_label.shape))

shape of train_data: (60000, 784)
shape of train_label: (60000,)


## Show the training data

In [6]:
# uncomment if you want to show the training data
#plt.figure(figsize=(20, 20))
#for index in range(10):
#    image = train_data[index+20000].reshape(28,28)
#    plt.subplot(2, 5, index+1)
#    plt.imshow(image)
#plt.show()

In [7]:
train_image_num = train_data.shape[0]
train_data = train_data.astype('float32')

print("train_image_num  is : {}".format(train_image_num))

train_image_num  is : 60000


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import time
import numpy as np

# =====================
# 假設 train_data / train_label 已經是 numpy array
# =====================
# 參數
EPOCH = 20
Batch_size = 500
Learning_rate = 5e-3
val_image_num = 10000  # 假設驗證集大小
train_image_num = train_data.shape[0]

# 資料預處理
train_data_tensor = torch.tensor(train_data[:train_image_num], dtype=torch.float32).view(-1, 1, 28, 28)/255.0
train_label_tensor = torch.tensor(train_label[:train_image_num], dtype=torch.long)

val_data_tensor = train_data_tensor[train_image_num - val_image_num:]
val_label_tensor = train_label_tensor[train_image_num - val_image_num:]

# =====================
# ConvNet 模型
# =====================
class ConvNet(nn.Module):
    def __init__(self, dropout_prob=0.25):
        super(ConvNet, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=0)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=0)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout_prob)
        self.fc1 = nn.Linear(32*24*24, 400)
        self.fc2 = nn.Linear(400, 100)
        self.fc3 = nn.Linear(100, 10)
        
    def forward(self, x):
        x = self.activation(self.conv1(x))
        x = self.activation(self.conv2(x))
        x = x.view(x.size(0), -1)  # flatten
        x = self.dropout(self.activation(self.fc1(x)))
        x = self.dropout(self.activation(self.fc2(x)))
        x = self.fc3(x)
        return x

net = ConvNet(dropout_prob=0.25)

# =====================
# Loss / Optimizer / Scheduler
# =====================
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=Learning_rate, momentum=0.9)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, eta_min=6e-5)  # 每5 epoch restart

# =====================
# Training Loop
# =====================
train_batch_num = (train_image_num - val_image_num) // Batch_size
val_batch_num = val_image_num // Batch_size

min_val_loss = torch.inf
min_val_loss_epoch = 0
acc = 0

for epoch in range(1, EPOCH+1):
    net.train()
    train_hit = 0
    total_train_loss = 0.0
    start_time = time.time()
    
    for it in range(train_batch_num):
        batch_X = train_data_tensor[it*Batch_size:(it+1)*Batch_size]
        batch_Y = train_label_tensor[it*Batch_size:(it+1)*Batch_size]
        
        optimizer.zero_grad()
        outputs = net(batch_X)
        loss = criterion(outputs, batch_Y)
        loss.backward()
        optimizer.step()
        
        _, pred_index = torch.max(outputs, 1)
        train_hit += (pred_index == batch_Y).sum().item()
        total_train_loss += loss.item()
    
    scheduler.step()  # 更新 learning rate

    # ===== Validation =====
    net.eval()
    val_hit = 0
    total_val_loss = 0.0
    with torch.no_grad():
        for it in range(val_batch_num):
            batch_X = val_data_tensor[it*Batch_size:(it+1)*Batch_size]
            batch_Y = val_label_tensor[it*Batch_size:(it+1)*Batch_size]
            
            outputs = net(batch_X)
            loss = criterion(outputs, batch_Y)
            
            _, pred_index = torch.max(outputs, 1)
            val_hit += (pred_index == batch_Y).sum().item()
            total_val_loss += loss.item()
    
    end_time = time.time()
    epoch_time = end_time - start_time

    print(f'Task-2  | Epoch:{epoch:3d} | Train Loss:{total_train_loss/train_batch_num:8.4f} | '
          f'Train Acc:{train_hit/(train_image_num-val_image_num)*100:6.2f}% | '
          f'Val Loss:{total_val_loss/val_batch_num:8.4f} | '
          f'Val Acc:{val_hit/val_image_num*100:6.2f}% | Epoch time:{epoch_time:5.2f} sec')
    
    if total_val_loss/val_batch_num < min_val_loss:
        print('Task2  | Minimum validation loss in the history. Saving model parameters...')
        torch.save(net.state_dict(), 'checkpoint.pth')
        min_val_loss = total_val_loss / val_batch_num
        min_val_loss_epoch = epoch
        acc = val_hit / val_image_num * 100.0

print(f'Task2  | Minimum validation loss: {min_val_loss:.4f} at Epoch {min_val_loss_epoch} | Val Acc: {acc:.2f}%')

Task-2  | Epoch:  1 | Train Loss:  1.8157 | Train Acc: 39.27% | Val Loss:  0.7954 | Val Acc: 70.28% | Epoch time:26.65 sec
Task2  | Minimum validation loss in the history. Saving model parameters...
Task-2  | Epoch:  2 | Train Loss:  0.7279 | Train Acc: 72.58% | Val Loss:  0.6120 | Val Acc: 76.75% | Epoch time:28.14 sec
Task2  | Minimum validation loss in the history. Saving model parameters...
Task-2  | Epoch:  3 | Train Loss:  0.6187 | Train Acc: 76.76% | Val Loss:  0.5577 | Val Acc: 78.63% | Epoch time:28.46 sec
Task2  | Minimum validation loss in the history. Saving model parameters...
Task-2  | Epoch:  4 | Train Loss:  0.5724 | Train Acc: 78.83% | Val Loss:  0.5344 | Val Acc: 79.16% | Epoch time:25.69 sec
Task2  | Minimum validation loss in the history. Saving model parameters...
Task-2  | Epoch:  5 | Train Loss:  0.5539 | Train Acc: 79.41% | Val Loss:  0.5187 | Val Acc: 81.09% | Epoch time:29.97 sec
Task2  | Minimum validation loss in the history. Saving model parameters...
Task-

## Change numpy array to pytorch tensor

In [9]:
# train_data_tensor = torch.from_numpy(train_data)
# train_label_tensor = torch.from_numpy(train_label)

# 資料預處理
# train_data_tensor = torch.tensor(train_data[:train_image_num], dtype=torch.float32).view(-1, 1, 28, 28)/255.0
# train_label_tensor = torch.tensor(train_label[:train_image_num], dtype=torch.long)            # (N,)


## Validation image number

In [10]:
# val_image_num=10000

# val_data_tensor = train_data_tensor[train_image_num-val_image_num:]  # last val_image_num as validation
# val_label_tensor = train_label_tensor[train_image_num-val_image_num:]

## Convert labels to one hot vector


In [11]:
# label_temp = np.zeros((train_image_num, 10), dtype = np.float32)
# for i in range(train_image_num):
#     label_temp[i][train_label[i]] = 1
# train_label_onehot = np.copy(label_temp)
# train_label_onehot_tensor = torch.from_numpy(train_label_onehot)
# print("One-hot training labels shape:",train_label_onehot.shape)


## Hyperparameters

In [12]:
# EPOCH = 100
# Batch_size = 100 # 10000 should be divisible by batch_size
# Learning_rate = 1e-2
# dropout_prob = 0.25

## Define the models with pytorch

In [13]:
# import torch.nn as nn
# import torch.nn.functional as F


# class ConvNet(nn.Module):
#     def __init__(self, dropout_prob=0.25):
#         super(ConvNet, self).__init__()
        
#         self.conv1 = nn.Conv2d(1, 16, 3)  # (16,26,26)
#         self.conv2 = nn.Conv2d(16, 32, 3) # (32,24,24)
#         self.activation = nn.ReLU()
#         self.dropout = nn.Dropout(dropout_prob)
#         self.fc1 = nn.Linear(24*24*32, 400)
#         self.fc2 = nn.Linear(400, 100)
#         self.fc3 = nn.Linear(100, 10)
        
#     def forward(self, x):
#         # 卷積層 + 激活
#         x = self.activation(self.conv1(x))
#         x = self.activation(self.conv2(x))
        
#         # flatten
#         x = x.view(x.size(0), -1)
        
#         # 全連接層 + 激活 + dropout
#         x = self.dropout(self.activation(self.fc1(x)))
#         x = self.dropout(self.activation(self.fc2(x)))
        
#         # 輸出層
#         x = self.fc3(x)
#         return x

# # 建立模型
# net = ConvNet(dropout_prob=0.25)
# # print(net)

# # net = Net()

## Criterion and Optimizer

In [14]:
# import torch.optim as optim

# criterion = nn.CrossEntropyLoss()
# optimizer = optim.SGD(net.parameters(), lr=Learning_rate, momentum=0.9)
# scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=100, eta_min=6e-5)

## Training

In [15]:
# import time

# # === 訓練與驗證迴圈 ===
# train_batch_num = train_data_tensor.size(0) // Batch_size
# val_batch_num = val_data_tensor.size(0) // Batch_size

# min_val_loss = torch.inf
# min_val_loss_epoch = 0
# acc = 0

# for epoch in range(1, EPOCH+1):
#     train_hit = 0
#     val_hit = 0
#     total_train_loss = 0.0
#     total_val_loss = 0.0
#     start_time = time.time()
    
#     net.train()
#     for it in range(train_batch_num):
#         optimizer.zero_grad()
#         batch_X = train_data_tensor[it*Batch_size:(it+1)*Batch_size]
#         batch_Y = train_label_tensor[it*Batch_size:(it+1)*Batch_size]
        
#         outputs = net(batch_X)
#         loss = criterion(outputs, batch_Y)
#         loss.backward()
#         optimizer.step()
        
#         _, pred_index = torch.max(outputs, 1)
#         train_hit += (pred_index == batch_Y).sum().item()
#         total_train_loss += loss.item()
    
#     scheduler.step()  # 每 epoch 更新一次 learning rate

#     net.eval()
    
#     with torch.no_grad():
#         for it in range(val_batch_num):
#             batch_X = val_data_tensor[it*Batch_size:(it+1)*Batch_size]
#             batch_Y = val_label_tensor[it*Batch_size:(it+1)*Batch_size]
#             outputs = net(batch_X)
#             loss = criterion(outputs, batch_Y)
#             _, pred_index = torch.max(outputs, 1)
#             val_hit += (pred_index == batch_Y).sum().item()
#             total_val_loss += loss.item()
    
#     end_time = time.time()
#     epoch_time = end_time - start_time
    
#     print(f'Task-2  | Epoch:{epoch:3d} | Train Loss:{total_train_loss/train_batch_num:8.4f} | '
#           f'Train Acc:{train_hit/(train_image_num-val_image_num)*100:6.2f}% | '
#           f'Val Loss:{total_val_loss/val_batch_num:8.4f} | '
#           f'Val Acc:{val_hit/val_image_num*100:6.2f}% | Epoch time:{epoch_time:5.2f} sec')
    
#     if total_val_loss/val_batch_num < min_val_loss:
#         print('Task2  | Minimum validation loss in the history. Saving model parameters...')
#         torch.save(net.state_dict(), 'checkpoint.pth')
#         min_val_loss = total_val_loss / val_batch_num
#         min_val_loss_epoch = epoch
#         acc = val_hit / val_image_num * 100.0

# print(f'Task2  | Minimum validation loss: {min_val_loss:.4f} at Epoch {min_val_loss_epoch} | Val Acc: {acc:.2f}%')